In [2]:
from PIL import Image
import numpy as np
import imageio
import os

In [18]:
dataset = "endonerf"
split = "pulling"
image_id = "00005"
n_iters = 3000
# dataset = "stereomis"
# split = "p2_1_1_247_s4"
# image_id = "00000"
# n_iters = 6000
out_dir = f"tmp/residual_map_{split}_{image_id}"
os.makedirs(out_dir, exist_ok=True)

mask_paths = [
    ["EndoGaussian", f"/network-volume/nhphuc/EndoFlow/output/{dataset}/{split}_alpha3d_0.25/test/ours_{n_iters}/masks/{image_id}.png"],
    ["Deform3DGS", f"/network-volume/nhphuc/EndoFlow/output/{dataset}/{split}_alpha3d_0.25/test/ours_{n_iters}/masks/{image_id}.png"],
    ["EndoPlanar", f"/network-volume/nhphuc/EndoFlow/output/{dataset}/{split}_alpha3d_0.25/test/ours_{n_iters}/masks/{image_id}.png"],
    ["FlowSplatting", f"/network-volume/nhphuc/EndoFlow/output/{dataset}/{split}_flow_alpha3d_0.0/test/ours_{n_iters}/masks/{image_id}.png"],
    ["EndoFlow", f"/network-volume/nhphuc/EndoFlow/output/{dataset}/{split}_alpha3d_0.25/test/ours_{n_iters}/masks/{image_id}.png"],
]
render_paths = [
    ["EndoGaussian", f"/network-volume/nhphuc/EndoGaussian/output/{dataset}/{split}/test/ours_{n_iters}/renders/{image_id}.png"],
    ["Deform3DGS", f"/network-volume/nhphuc/Deform3DGS/output/{dataset}/{split}/test/ours_{n_iters}/renders/{image_id}.png"],
    ["EndoPlanar", f"/network-volume/nhphuc/EndoPlanar/output/{dataset}/{split}/test/ours_{n_iters}/renders/{image_id}.png"],
    ["FlowSplatting", f"/network-volume/nhphuc/EndoFlow/output/{dataset}/{split}_flow_alpha3d_0.0/test/ours_{n_iters}/renders/{image_id}.png"],
    ["EndoFlow", f"/network-volume/nhphuc/EndoFlow/output/{dataset}/{split}_alpha3d_0.25/test/ours_{n_iters}/renders/{image_id}.png"],
]
gt_paths = [
    ["EndoGaussian", f"/network-volume/nhphuc/EndoFlow/output/{dataset}/{split}_alpha3d_0.25/test/ours_{n_iters}/gt/{image_id}.png"],
    ["Deform3DGS", f"/network-volume/nhphuc/EndoFlow/output/{dataset}/{split}_alpha3d_0.25/test/ours_{n_iters}/gt/{image_id}.png"],
    ["EndoPlanar", f"/network-volume/nhphuc/EndoFlow/output/{dataset}/{split}_alpha3d_0.25/test/ours_{n_iters}/gt/{image_id}.png"],
    ["FlowSplatting", f"/network-volume/nhphuc/EndoFlow/output/{dataset}/{split}_flow_alpha3d_0.0/test/ours_{n_iters}/gt/{image_id}.png"],
    ["EndoFlow", f"/network-volume/nhphuc/EndoFlow/output/{dataset}/{split}_alpha3d_0.25/test/ours_{n_iters}/gt/{image_id}.png"],
]

In [19]:
residual_maps = []

min_residual = float('inf')
max_residual = float('-inf')

for i in range(len(mask_paths)):
    mask = np.array(Image.open(mask_paths[i][1]).convert("L")) / 255.0
    render = np.array(Image.open(render_paths[i][1])) / 255.0
    gt = np.array(Image.open(gt_paths[i][1])) / 255.0

    residual_map = np.abs(render - gt) * mask[..., None]
    residual_map = np.mean(residual_map, axis=-1)
    residual_map = (residual_map * 255.0).astype(np.uint8)

    min_residual = min(min_residual, residual_map.min())
    max_residual = max(max_residual, residual_map.max())

    residual_maps.append(residual_map)

for i, res_map in enumerate(residual_maps):
    # res_map = (res_map - min_residual) / (max_residual - min_residual + 1e-8)
    # res_map = (res_map * 255.0).astype(np.uint8)
    imageio.imwrite(f"{out_dir}/residual_map_{split}_{image_id}_{mask_paths[i][0]}.png", res_map)